# Niche OT Demo

This standalone notebook demonstrates the current optimal-transport niche-library remodeling workflow with synthetic cluster-level data. It aligns a healthy/reference niche cluster library to a disease/target library using cell-type feature proportions (`Comp_*`) and tissue-mass proxies (`N_Cells`, plus reader-facing `tissue_mass`).

The helper module `transport_remodeling.py` is copied from the current research project. Real datasets, generated outputs, and the original duplicate organ notebooks are intentionally excluded.


## 1. Imports and Paths

The notebook is path-local so it can be inspected from the repository root without depending on the original project checkout.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import transport_remodeling as tr

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'transport_remodeling.py').exists():
    PROJECT_ROOT = Path(__file__).resolve().parent

DATA_DIR = PROJECT_ROOT / 'demo_data'
OUT_DIR = PROJECT_ROOT / 'demo_outputs'
OUT_DIR.mkdir(exist_ok=True)

healthy_path = DATA_DIR / 'healthy_cluster_library.csv'
disease_path = DATA_DIR / 'disease_cluster_library.csv'
healthy_path, disease_path


## 2. Load Synthetic Niche Libraries

Each row is a niche cluster. `Comp_*` columns are cell-type proportions; `N_Cells` is the mass used by the helper; `tissue_mass` is included to make the demo schema explicit.


In [ ]:
canonical_cell_map = {}

healthy_df = tr.load_cluster_metrics(healthy_path, canonical_map=canonical_cell_map)
disease_df = tr.load_cluster_metrics(disease_path, canonical_map=canonical_cell_map)

display(healthy_df)
display(disease_df)


In [ ]:
comp_cols = sorted(set(c for c in healthy_df.columns if c.startswith('Comp_')) | set(c for c in disease_df.columns if c.startswith('Comp_')))
schema_check = pd.DataFrame({
    'library': ['healthy', 'disease'],
    'n_clusters': [len(healthy_df), len(disease_df)],
    'mass_sum_from_N_Cells': [healthy_df['N_Cells'].sum(), disease_df['N_Cells'].sum()],
    'tissue_mass_sum': [healthy_df['tissue_mass'].sum(), disease_df['tissue_mass'].sum()],
    'n_cell_type_features': [len(comp_cols), len(comp_cols)],
})
schema_check


## 3. Build Composition Costs and Masses

The current helper computes normalized cell-type composition matrices, scaled cosine-distance costs, and normalized cluster masses.


In [ ]:
options = tr.TransportOptions(
    mass_mode='n_cells',
    lambda_expand=0.35,
    lambda_reduce=0.35,
    lambda_residual=0.80,
    branch_min_total_mass=0.01,
    branch_min_source_fraction=0.10,
    top_n_heatmap=10,
)

healthy_comp, disease_comp, comp_cols = tr.aligned_composition_matrices(healthy_df, disease_df)
healthy_mass, disease_mass = tr.compute_masses(healthy_df, disease_df, options)
total_cost, cost_long, cost_metadata = tr.build_cost_matrices(
    healthy_comp, disease_comp, healthy_df, disease_df, options
)

cost_long.sort_values('total_cost').head(8)


## 4. Solve Unbalanced Transport

The solver links healthy-source clusters to disease-target clusters while allowing expansion, reduction, and residual target mass.


In [ ]:
solution = tr.solve_unbalanced_transport(
    total_cost,
    healthy_mass,
    disease_mass,
    lambda_expand=options.lambda_expand,
    lambda_reduce=options.lambda_reduce,
    lambda_residual=options.lambda_residual,
)

tr.objective_components(
    solution,
    total_cost,
    options.lambda_expand,
    options.lambda_reduce,
    options.lambda_residual,
)


## 5. Summarize Remodeling Events

The same helper functions generate source summaries, target summaries, and a long transport plan table for downstream review.


In [ ]:
current_summary = tr.summarize_current_best_match(
    cost_long, healthy_df, disease_df, healthy_mass, disease_mass, options.pseudocount_mass
)
source_summary = tr.summarize_sources(
    solution, total_cost, healthy_df, disease_df, healthy_mass, options, current_summary=current_summary
)
target_summary = tr.summarize_targets(
    solution, total_cost, healthy_df, disease_df, disease_mass, options
)
plan_df = tr.build_plan_df(
    solution, cost_long, healthy_df, disease_df, healthy_mass, disease_mass, total_cost, options
)

display(source_summary[['healthy_cluster_id', 'healthy_mass', 'transported_out_mass', 'net_shift_mass', 'event_labels', 'top_disease_targets']])
display(target_summary[['disease_cluster_id', 'disease_mass', 'explained_transport_mass', 'residual_mass', 'dominant_healthy_source', 'high_unknown_flag']])
display(plan_df[['healthy_cluster_id', 'disease_cluster_id', 'transport_mass', 'total_cost', 'composition_similarity']])


## 6. Optional Demo Outputs

The repository ignores `demo_outputs/`. These writes are illustrative only and are not required for reviewing the code.


In [ ]:
source_summary.to_csv(OUT_DIR / 'source_summary.csv', index=False)
target_summary.to_csv(OUT_DIR / 'target_summary.csv', index=False)
plan_df.to_csv(OUT_DIR / 'transport_plan.csv', index=False)

tr.plot_transport_heatmap(solution, healthy_df, disease_df, OUT_DIR / 'transport_heatmap.png', top_n=options.top_n_heatmap)
tr.write_json(OUT_DIR / 'cost_metadata.json', cost_metadata)

sorted(p.name for p in OUT_DIR.iterdir())
